# 18b · CP6 analysis after Phase D (CPU runtime, 0 GPU)

**CPU runtime.** About **0.05 CU** at 0.08 CU/h (estimate). No model is loaded: this notebook reads what notebook 18 saved, plus the evidence bundles and the P6 image tar for the ground truth. Dhanush's review of Phase D (25 Nov):

1. **Template bug.** In Phase D the fixed template failed the verifier on 91 of 1,500 reports. Here the failing rule is counted, and the fixed template (it skips an E region with mask_conf < 0.6, as the LLM must) is scored beside the as-run one.
2. **HR-GT split** for every variant: the verdict vs the region claims (wrong / checkable each), on P6 and P6_mllm, with and without the 26 flagged SID-Set masks. The region-only comparison of the grounded reports and the image baseline is the headline row.
3. **Image baseline:** the 3x3 verdict confusion matrix on P6_mllm, its boxes by claim type, and a sensitivity HR-GT that scores every boxed claim whatever its type (the pre-registered rule stays primary).
4. **Limitation claims** that failed, counted by rule.
5. **CP6 criterion 1:** every claim of every final report passes all rules. **Criterion 2:** the 30 reports to read (the same 30 as notebook 18), each claim with its verifier errors, for Dhanush's read beside the images.

The as-run numbers are recomputed and checked against `cp6_reports_seed42.json`, so this notebook scores exactly what notebook 18 scored.

**Results** (send to Claude): `Paper 1/runs/cp6/cp6_analysis_seed42.json`, `cp6_read30_seed42.json` and `compute_cp6_analysis.json`, plus the balance before and after if Phase C (notebook 17) runs at the same time.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings and pre-flight
import datetime, json, random, shutil, tempfile, time
from pathlib import Path
from eviground.env import get_paths, session_info
from eviground.eval import p6cache
from eviground.explain import cp6analysis, evidence, llm, variants, verifier
from eviground.utils import session

SEED = 42
CU_BALANCE_START = None     # optional on this CPU runtime (about 0.05 CU, estimate)
EST_CU = 0.05
N_READ = 30                 # the 30 reports notebook 18 picked (same seed, same rule)
T0 = time.time()
info = session_info(cu_per_hour=session.CU_RATES['cpu'])
paths = get_paths()
if DRY:
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_cp6b_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    N_READ, EST_CU = 2, 0.0
paths.ensure()
CK = paths.runs / 'checkpoints'
SUBSETS_FILE = CK / 'eval_subsets.json'
BUNDLES = paths.runs / 'explain' / f'p6_unified_ctx_seed{SEED}'
TAR_NAME = 'p6_images.tar'
CACHE_TAR, CACHE_MAN = paths.runs / 'cache' / TAR_NAME, paths.runs / 'cache' / 'p6_images_manifest.json'
OUT = paths.runs / 'cp6'
OUT.mkdir(parents=True, exist_ok=True)
GROUNDED_JSONL, IMAGE_JSONL = OUT / f'p6_grounded_reports_seed{SEED}.jsonl', OUT / f'p6_image_baseline_seed{SEED}.jsonl'
PHASE_D = OUT / f'cp6_reports_seed{SEED}.json'
RESULT = OUT / f'cp6_analysis_seed{SEED}.json'
READ_FILE = OUT / f'cp6_read30_seed{SEED}.json'

In [ ]:
# Cell 3: in the dry run, a fake P6 and notebook 18's outputs (MockLLM / MockVL); then the pre-flight
if DRY:
    import numpy as np
    from eviground.eval import p6data, subsets
    from eviground.eval.tiled import Item
    rng = np.random.default_rng(0)
    items, part_of = [], {}
    DRY_PROBS = {'edited': {'real': 0.05, 'generated': 0.05, 'edited': 0.9},
                 'real': {'real': 0.8, 'generated': 0.15, 'edited': 0.05},
                 'generated': {'real': 0.1, 'generated': 0.85, 'edited': 0.05}}
    for i in range(6):
        part = ['diffseg_heldout', 'sidset_tampered', 'brgen', 'real', 'generated', 'sidset_tampered'][i]
        img = (rng.random((96, 128, 3)) * 255).astype(np.uint8)
        edited = part in p6data.EDITED_PARTS
        mask = np.zeros((96, 128), bool)
        mask[20:60, 30:90] = True
        meta = {'part': part, 'label': 'edited' if edited else part}
        if i == 5:   # an aspect-mismatched SID-Set mask (flagged, as notebook 15 found 26)
            meta.update(mask_resized_from=[60, 128], mask_aspect_mismatch=True)
        uid = f'dry/{part}/{i}.png'
        items.append(Item(uid, img, mask if edited else None, None, meta)); part_of[uid] = part
        probs = DRY_PROBS['edited' if edited else part]
        mp = np.zeros((96, 128)); mp[20:60, 30:90] = (0.55 if i == 2 else 0.9) * edited   # i == 2: below tau
        rec, rles = evidence.evidence_record(probs, 0.6, mp, mp + rng.random((96, 128)) * 0.1, img)
        evidence.save_bundle(BUNDLES, uid, rec, rles, {'decoder': mp}, probs, 0.6)
    p6data.write_cache(items, CACHE_TAR, CACHE_MAN)
    chosen = [{'uid': u} for u in part_of]
    subsets.save({'P6': subsets._record('P6', chosen, SEED, 'dry', part_of=part_of),
                  'P6_mllm': subsets._record('P6_mllm', chosen[:4], SEED, 'dry')}, SUBSETS_FILE)
    # notebook 18's outputs, written by its own code paths with the mocks (one injected wrong number)
    _recs = [(it.uid, evidence.load_bundle(BUNDLES, it.uid)['evidence']) for it in items]
    with open(GROUNDED_JSONL, 'w') as fh:
        for r in verifier.run_grounded(llm.MockLLM({1: 'bad_number'}), _recs):
            fh.write(json.dumps(r, default=str) + '\n')
    _mllm = items[:4]
    with open(IMAGE_JSONL, 'w') as fh:
        for it, r in zip(_mllm, variants.image_reports(llm.MockVL(), [it.image for it in _mllm])):
            fh.write(json.dumps({'uid': it.uid, **r}) + '\n')
GPU = info.get('gpu_name')
PREFLIGHT = session.preflight(
    {'notebook': '18b CP6 analysis (CPU, no model)', 'runtime': GPU or 'cpu', 'bundles': BUNDLES, 'result file': RESULT},
    paths={'evidence bundles (notebook 14)': BUNDLES, 'P6 image tar': CACHE_TAR, 'P6 tar manifest': CACHE_MAN,
           'eval_subsets.json': SUBSETS_FILE, 'grounded reports (notebook 18)': GROUNDED_JSONL,
           'image-baseline reports (notebook 18)': IMAGE_JSONL},
    est_cu=EST_CU, cu_left=CU_BALANCE_START)
STORED = json.loads(SUBSETS_FILE.read_text())
P6, P6_MLLM = STORED['P6'], STORED['P6_mllm']
LOCAL = paths.local_data / 'p6_cache'
LOCAL_TAR = paths.local_data / TAR_NAME
shutil.copyfile(CACHE_TAR, LOCAL_TAR)
CACHE = p6cache.read_cache(LOCAL_TAR, CACHE_MAN, LOCAL)
RECORDS = {u: evidence.load_bundle(BUNDLES, u)['evidence'] for u in P6['uids']}
print(f"P6 {len(RECORDS)} evidence records, cache {CACHE['images']} images | P6_mllm {P6_MLLM['images']}")

In [ ]:
# Cell 4: notebook 18's reports and the ground truth (sizes, masks, metadata; flagged SID-Set masks)
GROUNDED = {r['uid']: r for r in (json.loads(l) for l in GROUNDED_JSONL.read_text().splitlines() if l.strip())}
IMAGE = {r['uid']: r for r in (json.loads(l) for l in IMAGE_JSONL.read_text().splitlines() if l.strip())}
assert set(GROUNDED) == set(P6['uids']), 'grounded reports are incomplete'
assert set(IMAGE) == set(P6_MLLM['uids']), 'image-baseline reports are incomplete'
GT = {}
for x in p6cache.cached_arrays(P6['uids'], CACHE, LOCAL):
    GT[x['uid']] = {'shape': x['image'].shape[:2], 'mask': x['mask'], 'meta': x['meta']}
    del x
FLAGGED = {u for u, x in GT.items() if p6cache.is_aspect_mismatched(x['meta'], x['shape'])}
LABEL = {u: GT[u]['meta']['label'] for u in P6['uids']}
print(f'{len(GROUNDED)} grounded, {len(IMAGE)} image-baseline reports | {len(FLAGGED)} flagged SID-Set masks')

In [ ]:
# Cell 5: items 1-5 of the review
# 1. the template, as run in Phase D and fixed
TEMPLATE_AS_RUN = {u: variants.template_report(RECORDS[u], min_mask_conf=None) for u in P6['uids']}
TEMPLATE_FIXED = {u: variants.template_report(RECORDS[u]) for u in P6['uids']}
TEMPLATE_RULES = {name: cp6analysis.rule_counts((t[u], RECORDS[u]) for u in P6['uids'])
                  for name, t in (('as_run', TEMPLATE_AS_RUN), ('fixed', TEMPLATE_FIXED))}
print('template, as run:', TEMPLATE_RULES['as_run'])
print('template, fixed :', TEMPLATE_RULES['fixed'])
assert TEMPLATE_RULES['fixed']['failing_claims'] == 0, 'the fixed template still fails the verifier'

# 2-3. HR-GT per variant, split into the verdict and the region claims; the image-baseline sensitivity
def _rows(uids, reports, with_record=True, include_flagged=False):
    for u in uids:
        if u in FLAGGED and not include_flagged:
            continue
        yield reports[u], LABEL[u], GT[u]['mask'], (RECORDS[u] if with_record else None)

VARIANTS = {
    'grounded_final': {u: GROUNDED[u]['final_report'] for u in P6['uids']},
    'no_verifier': {u: GROUNDED[u]['first_report'] for u in P6['uids']},
    'template_as_run': TEMPLATE_AS_RUN,
    'template_fixed': TEMPLATE_FIXED,
}
PIXEL = {u: IMAGE[u]['pixel_report'] for u in P6_MLLM['uids']}
HRGT = {}
for scope, uids in (('P6', P6['uids']), ('P6_mllm', P6_MLLM['uids'])):
    for inc in (False, True):
        key = f"{scope}{'_incl_mismatched' if inc else ''}"
        HRGT[key] = {name: verifier.hr_gt_summary(_rows(uids, reps, include_flagged=inc)) for name, reps in VARIANTS.items()}
        if scope == 'P6_mllm':
            HRGT[key]['image_baseline'] = verifier.hr_gt_summary(_rows(uids, PIXEL, with_record=False, include_flagged=inc))
            HRGT[key]['image_baseline_all_boxes'] = verifier.hr_gt_summary(
                _rows(uids, PIXEL, with_record=False, include_flagged=inc), score_all_boxes=True)
HEADLINE = {key: {name: HRGT[key][name]['region_claims'] for name in ('grounded_final', 'image_baseline', 'image_baseline_all_boxes')}
            for key in ('P6_mllm', 'P6_mllm_incl_mismatched')}
for key, rows in HRGT.items():
    for name, s in rows.items():
        v, r = s['verdict_claims'], s['region_claims']
        print(f"{key:24} {name:24} HR-GT {s['hr_gt']:.4f} | verdict {v['wrong']}/{v['checkable']} | "
              f"region {r['wrong']}/{r['checkable']}" + (f" = {r['rate']:.4f}" if r['rate'] is not None else ''))

# the as-run numbers must reproduce notebook 18's result file exactly
REPRODUCED = None
if PHASE_D.is_file():
    _d = json.loads(PHASE_D.read_text())['hr_gt']
    _map = {'grounded_final': 'grounded_final', 'no_verifier': 'no_verifier', 'template_as_run': 'template', 'image_baseline': 'image_baseline'}
    _keys = ('images', 'checkable_claims', 'wrong_claims', 'claims_without_region', 'verdict_accuracy')
    REPRODUCED = all(HRGT[k][ours][f] == _d[k][theirs][f] for k in _d for ours, theirs in _map.items()
                     if theirs in _d[k] for f in _keys)
    print('as-run HR-GT reproduces notebook 18:', REPRODUCED)
    if not DRY:
        assert REPRODUCED, 'the recomputed as-run HR-GT differs from notebook 18'

# 3. the image baseline's verdicts and boxes
CONFUSION = {
    'image_baseline': cp6analysis.verdict_confusion((IMAGE[u]['report'], LABEL[u]) for u in P6_MLLM['uids']),
    'detector_verdict_p6_mllm': cp6analysis.verdict_confusion((GROUNDED[u]['final_report'], LABEL[u]) for u in P6_MLLM['uids']),
    'detector_verdict_p6': cp6analysis.verdict_confusion((GROUNDED[u]['final_report'], LABEL[u]) for u in P6['uids']),
}
BOXES_BY_TYPE = cp6analysis.boxes_by_claim_type(IMAGE[u]['report'] for u in P6_MLLM['uids'])
print('image-baseline verdicts (true -> said):', CONFUSION['image_baseline'])
print('image-baseline claims by type:', BOXES_BY_TYPE)

# 4. limitation claims that failed, by rule (first pass)
FIRST = {'P6': verifier.first_pass_metrics(GROUNDED[u] for u in P6['uids']),
         'P6_mllm': verifier.first_pass_metrics(GROUNDED[u] for u in P6_MLLM['uids'])}
for k, m in FIRST.items():
    print(k, 'limitation claims', m['limitation_claims'], 'failed by rule', m['limitation_rejections_by_rule'])

# 5. CP6 criterion 1 (every final claim passes all rules); the 30 reports to read (criterion 2)
FINAL_CHECK = cp6analysis.final_check(GROUNDED.values(), RECORDS)
print('criterion 1:', FINAL_CHECK)
READ = sorted(random.Random(SEED).sample(P6['uids'], min(N_READ, len(P6['uids']))))   # notebook 18's rule
READ_ENTRIES = [cp6analysis.read_entry(u, RECORDS[u], GROUNDED[u], LABEL[u], IMAGE.get(u)) for u in READ]

In [ ]:
# Cell 6: result files, compute record, release
READ_FILE.write_text(json.dumps(READ_ENTRIES, indent=1, default=str))
RESULT_DATA = {
    'step': 'CP6 analysis after Phase D (Dhanush review, 25 Nov)', 'dry_run': DRY, 'branch': BRANCH, 'seed': SEED,
    'session': info, 'inputs': {'grounded': GROUNDED_JSONL.name, 'image_baseline': IMAGE_JSONL.name, 'phase_d': PHASE_D.name},
    'as_run_reproduces_phase_d': REPRODUCED, 'flagged_sidset_masks': len(FLAGGED),
    'template': TEMPLATE_RULES, 'hr_gt': HRGT, 'headline_region_only': HEADLINE,
    'verdict_confusion': CONFUSION, 'image_baseline_claims_by_type': BOXES_BY_TYPE,
    'limitation_claims': {k: {'counts': m['limitation_claims'], 'failed_by_rule': m['limitation_rejections_by_rule']}
                          for k, m in FIRST.items()},
    'criterion_1_final_reports': FINAL_CHECK, 'read30_uids': READ,
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
}
RESULT.write_text(json.dumps(RESULT_DATA, indent=1, default=str))
print('saved', RESULT, 'and', READ_FILE)
COMPUTE = session.finish(OUT, T0, gpu=GPU or 'cpu', name='compute_cp6_analysis.json', balance_start=CU_BALANCE_START,
                         extra={'git_commit': info['git_commit'], 'est_cu': EST_CU})
if DRY:
    assert TEMPLATE_RULES['as_run']['failing_claims_by_rule'] == {'c': 1}   # the dry E region below tau
    assert TEMPLATE_RULES['as_run']['reports_passing'] == len(P6['uids']) - 1
    assert TEMPLATE_RULES['fixed']['reports_passing'] == len(P6['uids'])
    assert REPRODUCED is None   # notebook 18's result file is not in this dry world
    assert FINAL_CHECK['passes'], FINAL_CHECK
    for k in HRGT:
        for s in HRGT[k].values():
            assert s['verdict_claims']['checkable'] + s['region_claims']['checkable'] == s['checkable_claims']
            assert s['verdict_claims']['wrong'] + s['region_claims']['wrong'] == s['wrong_claims']
    assert len(READ_ENTRIES) == N_READ and set(HEADLINE) == {'P6_mllm', 'P6_mllm_incl_mismatched'}
RELEASED = session.release(DRY)
if DRY:
    assert RELEASED == 'dry', RELEASED